# Export a saved LoRA adapter to GGUF (recovery notebook)

Use this when training already finished and the adapter folder `lora_<RUN_NAME>` is in your Google Drive, but the exported
GGUF did not contain the fine-tuning (it answered like the plain base model in Ollama).

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Run the cells **one at a time**. The *verify* cell stops with an error if the loaded model is not fine-tuned - in that
   case do not export; send me its output.
3. The GGUF and a `Modelfile` are copied to a **new** Drive folder (`export_<RUN_NAME>_v2`) so the old file is not overwritten.
4. Import it into Ollama and test it (last cell).

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

RUN_NAME   = "llama32_3b"
DRIVE_DIR  = "/content/drive/MyDrive/bp_finetune"      # folder the training notebook copied to
ADAPTER_DIR = f"{DRIVE_DIR}/lora_{RUN_NAME}"
NO_SYSTEM_ROLE = False                                 # set True only for Gemma (its template has no system role)
SYSTEM = ("You are a health-information assistant for older adults. Give short, plain "
          "answers. Do not diagnose or prescribe medication.")

print(sorted(os.listdir(ADAPTER_DIR)))
size_mb = os.path.getsize(f"{ADAPTER_DIR}/adapter_model.safetensors") / 1e6
print(f"adapter_model.safetensors = {size_mb:.1f} MB  (expect roughly 100-200 MB for a 3B model with r=16)")
assert size_mb > 20, "adapter file is tiny - training probably did not save the adapter"

### Load the adapter
Pointing `from_pretrained` at the adapter folder loads the base model it was trained from, attaches the adapter and
reads the saved tokenizer / chat template.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

def build_messages(question):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    if NO_SYSTEM_ROLE:
        msgs = [{"role": "user", "content": SYSTEM + "\n\n" + question}]
    return msgs

def ask(question, max_new_tokens = 200):
    ids = tokenizer.apply_chat_template(build_messages(question), tokenize = True,
                                        add_generation_prompt = True, return_tensors = "pt").to("cuda")
    out = model.generate(input_ids = ids, max_new_tokens = max_new_tokens, use_cache = True,
                         temperature = 0.1, min_p = 0.1)
    return tokenizer.batch_decode(out[:, ids.shape[1]:], skip_special_tokens = True)[0]

### Verify the model is really fine-tuned
Expected: answers start with `Blood pressure status:`; 106/49 should be **Low**, 145/92 **High**, 176/130 **High (possible emergency)**.
The cell raises an error if the format is missing.

In [ ]:
tests = ["My blood pressure is 145/92. Is that okay?",
         "My blood pressure is 106/49. Is that okay?",
         "I just measured 176 over 130 on my home monitor. What does that mean?",
         "Is a blood pressure of 118/76 considered high? Answer Yes or No."]
answers = [ask(q) for q in tests]
for q, a in zip(tests, answers):
    print("Q:", q, "\nA:", a[:300], "\n" + "-" * 60)
assert answers[0].strip().startswith("Blood pressure status"), (
    "NOT fine-tuned (no 'Blood pressure status' format). Do not export - send me this output.")
print("OK: the loaded model is fine-tuned. Safe to export.")

### Export (only run this if the verify cell printed OK)

In [ ]:
model.save_pretrained_gguf(f"bp_{RUN_NAME}", tokenizer, quantization_method = "q4_k_m")

In [ ]:
!ls -lh bp_{RUN_NAME}*

### Copy to a new Drive folder
Also lowers the Modelfile's `temperature 1.5` (Unsloth's default, far too random for advice) to `0.3`.

In [ ]:
import glob, re, shutil
OUT_DIR = f"{DRIVE_DIR}/export_{RUN_NAME}_v2"
os.makedirs(OUT_DIR, exist_ok = True)
for folder in glob.glob(f"bp_{RUN_NAME}*"):
    for path in glob.glob(f"{folder}/*"):
        if os.path.isfile(path):
            shutil.copy(path, OUT_DIR)
modelfile = f"{OUT_DIR}/Modelfile"
if os.path.exists(modelfile):
    text = open(modelfile).read()
    open(modelfile, "w").write(re.sub(r"PARAMETER temperature [0-9.]+", "PARAMETER temperature 0.3", text))
print(sorted(os.listdir(OUT_DIR)))

### On your PC: import into Ollama and test
Download the folder `export_<RUN_NAME>_v2` from Drive, open a terminal **inside it**, then:

    ollama create bp-llama32 -f Modelfile
    ollama run bp-llama32

Ask: `My blood pressure is 106/49. Is that okay?` - the answer must start with `Blood pressure status: Low`.
If it answers like the plain base model again, send me the output of the verify cell above.